# Question Answering with LangChain, OpenAI, and MultiQuery Retriever

This interactive workbook demonstrates example of Elasticsearch's [MultiQuery Retriever](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) to generate similar queries for a given user input and apply all queries to retrieve a larger set of relevant documents from a vectorstore.

Before we begin, we first split the fictional workplace documents into passages with `langchain` and uses OpenAI to transform these passages into embeddings and then store these into Elasticsearch.

We will then ask a question, generate similar questions using langchain and OpenAI, retrieve relevant passages from the vector store, and use langchain and OpenAI again to provide a summary for the questions.

## Install packages and import modules

In [1]:
!pip install -qU "langchain>=1.0" "langchain-core>=0.3" "langchain-community>=0.4" "langchain-classic>=0.3" langchain-openai langchain-elasticsearch tiktoken jq lark elasticsearch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.2/502.2 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.8/773.8 kB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.8/952.8 kB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==

In [2]:
import os
from getpass import getpass
from langchain_openai.embeddings import OpenAIEmbeddings
#from langchain_elasticsearch import ElasticsearchStore
from langchain_openai import ChatOpenAI
from langchain_classic.retrievers.multi_query import MultiQueryRetriever  # ← CORRECT import for 1.0+


from langchain_community.vectorstores.elasticsearch import ElasticsearchStore
#from langchain_openai import OpenAIEmbeddings

# os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

## Connect to Elasticsearch

ℹ️ We're using an Elastic Cloud deployment of Elasticsearch for this notebook. If you don't have an Elastic Cloud deployment, sign up [here](https://cloud.elastic.co/registration?utm_source=github&utm_content=elasticsearch-labs-notebook) for a free trial.

We'll use the **Cloud ID** to identify our deployment, because we are using Elastic Cloud deployment. To find the Cloud ID for your deployment, go to https://cloud.elastic.co/deployments and select your deployment.

We will use [ElasticsearchStore](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html) to connect to our elastic cloud deployment, This would help create and index data easily.  We would also send list of documents that we created in the previous step

In [12]:
# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#finding-your-cloud-id
ELASTIC_CLOUD_ID = getpass("Elastic Cloud ID: ")

# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#creating-an-api-key
ELASTIC_API_KEY = getpass("Elastic Api Key: ")

# https://platform.openai.com/api-keys
OPENAI_API_KEY = getpass("OpenAI API key: ")


# Create OpenAI embedding model
embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

# Set a meaningful index name
INDEX_NAME = "lab-multiquery-rag-20260228" #give it a meaningful name,

# Create Elasticsearch vector store
vectorstore = ElasticsearchStore(
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
    index_name=INDEX_NAME,
    embedding=embeddings  )


Elastic Cloud ID: ··········
Elastic Api Key: ··········
OpenAI API key: ··········


In [11]:
print("Cloud ID has colon:", ":" in ELASTIC_CLOUD_ID)
print("Cloud ID length:", len(ELASTIC_CLOUD_ID))
print("Cloud ID first 20 chars:", ELASTIC_CLOUD_ID[:20])

Cloud ID has colon: True
Cloud ID length: 165
Cloud ID first 20 chars: 0a01bf1c0293407c8de3


## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [13]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [15]:
from langchain_community.document_loaders import JSONLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import json  # Optional: for metadata extraction
from datetime import datetime # Import datetime here

def metadata_func(record: dict, metadata: dict | None = None) -> dict:
    """
    Build clean metadata for each document.

    Expected output keys:
    - name
    - summary
    - url
    - category
    - updated_at
    """
    metadata = dict(metadata or {})

    # 1) Find the "source" dict safely (depends on how the loader shaped the record)
    # Common patterns: record itself is the source, or record["source"]
    source = record.get("source") if isinstance(record, dict) else None
    if not isinstance(source, dict):
        source = record if isinstance(record, dict) else {}

    # 2) Helper: first non-empty value from candidates
    def pick(*candidates):
        for c in candidates:
            if c is None:
                continue
            if isinstance(c, str) and c.strip() == "":
                continue
            return c
        return None

    # 3) Populate required fields with best-effort fallbacks
    name = pick(
        source.get("name"),
        source.get("title"),
        source.get("document_title"),
        record.get("name") if isinstance(record, dict) else None,
        record.get("title") if isinstance(record, dict) else None,
        "unknown",
    )

    summary = pick(
        source.get("summary"),
        source.get("description"),
        source.get("text_summary"),
        record.get("summary") if isinstance(record, dict) else None,
        "",
    )

    url = pick(
        source.get("url"),
        source.get("source_url"),
        source.get("link"),
        record.get("url") if isinstance(record, dict) else None,
        "",
    )

    category = pick(
        source.get("category"),
        source.get("tag"),
        source.get("type"),
        record.get("category") if isinstance(record, dict) else None,
        "unknown",
    )

    updated_at = pick(
        source.get("updated_at"),
        source.get("last_updated"),
        source.get("timestamp"),
        record.get("updated_at") if isinstance(record, dict) else None,
        "",
    )

    metadata.update(
        {
            "name": str(name),
            "summary": str(summary),
            "url": str(url),
            "category": str(category),
            "updated_at": str(updated_at),
        }
    )

    return metadata


loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",  # Extracts array of records
    content_key="content",
    metadata_func=metadata_func,
)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=1000,       # e.g., ~750 words
    chunk_overlap=200,     # Overlap for context preservation
)

docs = loader.load_and_split(text_splitter=text_splitter)


### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [18]:
from datetime import datetime
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

clean_docs = []
required_keys = ["name", "summary", "url", "category", "updated_at"]

for doc in docs:
    metadata = doc.metadata.copy()

    for key in required_keys:
        if metadata.get(key) in [None, "null"]:
            metadata[key] = ""

    raw_updated_at = metadata.get("updated_at")

if raw_updated_at is None:
    metadata["updated_at"] = datetime.now().isoformat()
else:
    raw_str = str(raw_updated_at).strip().lower()
    if raw_str in {"", "none", "null"}:
        metadata["updated_at"] = datetime.now().isoformat()
    else:
        # keep original if it already looks like a timestamp
        metadata["updated_at"] = str(raw_updated_at).strip()

    clean_docs.append(doc.model_copy(update={"metadata": metadata}))
# Create embeddings ONCE
embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

# Create vectorstore ONCE using cleaned docs
vectorstore = ElasticsearchStore.from_documents(
    clean_docs,
    embeddings,
    index_name=INDEX_NAME,
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)

# Create LLM
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0,
    openai_api_key=OPENAI_API_KEY # Explicitly pass the API key
)

# Create MultiQueryRetriever
retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 4}),
    llm=llm
)


# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [19]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import chain as lc_chain
import logging

# Enable detailed logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Multi-query generator with 3 variants
MULTI_QUERY_PROMPT = ChatPromptTemplate.from_template("""
Generate 3 diverse versions of this question for better retrieval. Vary phrasing, keywords, and perspectives:

{question}

Queries (one per line):
""")

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template("""
📄 [{source}]
{page_content}
---
""")

# Define the context prompt for the LLM
LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:
{context}

Question: {question}
""")

def safe_combine_docs(docs):
    """Production-ready doc formatting with fallbacks"""
    doc_strings = []
    for i, doc in enumerate(docs):
        try:
            doc_dict = doc.model_dump()
            source = doc.metadata.get("name") or doc.metadata.get("source", f"Doc-{i}")
            doc_dict["source"] = source
            formatted = LLM_DOCUMENT_PROMPT.format(**doc_dict)
        except Exception as e:
            logger.warning(f"Doc format error: {e}")
            formatted = f"[Doc-{i}] {doc.page_content[:500]}..."
        doc_strings.append(formatted)
    return "\n\n".join(doc_strings)

# Self-healing chain: retry bad retrievals
def self_healing_retriever(query, max_tries=2):
    """Retry with rewritten query if empty results"""
    for attempt in range(max_tries):
        docs = retriever.invoke(query)
        if docs:
            return docs
        logger.info(f"Empty results (attempt {attempt+1}), rewriting...")
        query = llm.invoke(f"Rewrite for better retrieval: {query}").content
    return retriever.invoke(query)  # Fallback

_context = RunnableParallel(
    context=(RunnablePassthrough() | self_healing_retriever | safe_combine_docs),
    question=RunnablePassthrough(),
)

rag_chain = _context | LLM_CONTEXT_PROMPT | llm | StrOutputParser()

# Test with auto-multi-query
def multi_query_rag(question):
    """Generate + retrieve + answer"""

    query_chain = MULTI_QUERY_PROMPT | llm | StrOutputParser()

    generated_queries = query_chain.invoke({"question": question})

    print("\nGenerated Queries:")
    print("------------------")
    print(generated_queries)
    print("------------------\n")

    queries = [q.strip() for q in generated_queries.split("\n") if q.strip()]

    all_docs = []

    for q in queries:
        docs = self_healing_retriever(q)
        all_docs.extend(docs[:3])

    return rag_chain.invoke({
        "question": question,
        "context": safe_combine_docs(all_docs)
    })

print("---- Answer ----")

print(multi_query_rag("what is the nasa sales team?"))

---- Answer ----

Generated Queries:
------------------
1. Can you provide information about the sales team at NASA?
2. What is the role of the sales team within NASA?
3. How does NASA's sales team operate and contribute to the organization's goals?
------------------

The context provided does not contain any information about the NASA sales team.


**Generate at least two new iteratioins of the previous cells - Be creative.** Did you master Multi-
Query Retriever concepts through this lab?

## Side-by-side comparison (Single Query vs Multi-Query)





In [20]:
# =========================
# Iteration 1: Single-query vs Multi-query comparison
# =========================

from langchain_classic.retrievers.multi_query import MultiQueryRetriever

# Base retriever (single query)
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 6})

# MultiQuery retriever (expands queries internally)
mq_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm
)

def print_sources(docs, label, max_items=8):
    print(f"\nSources ({label}) - top {min(max_items, len(docs))}:")
    for i, d in enumerate(docs[:max_items], start=1):
        name = d.metadata.get("name") or d.metadata.get("source") or f"Doc-{i}"
        category = d.metadata.get("category") or "unknown"
        updated_at = d.metadata.get("updated_at") or "unknown"
        print(f"{i}. {name} | {category} | {updated_at}")

def answer_with_retriever(question: str, retriever_obj, label: str) -> str:
    docs = retriever_obj.invoke(question)
    print(f"\n--- {label} ---")
    print(f"Retrieved docs: {len(docs)}")
    print_sources(docs, label=label)

    context = safe_combine_docs(docs)
    return rag_chain.invoke({"question": question, "context": context})

question = "Write a short summary of the most important points about <PUT A REAL TOPIC FROM YOUR DATA HERE>."

print("\n==============================")
print("Iteration 1: Comparison run")
print("==============================")

single_answer = answer_with_retriever(question, base_retriever, "Single Query")
multi_answer = answer_with_retriever(question, mq_retriever, "Multi Query")

print("\n\n--- Single Query Answer ---")
print(single_answer)

print("\n\n--- Multi Query Answer ---")
print(multi_answer)


Iteration 1: Comparison run

--- Single Query ---
Retrieved docs: 6

Sources (Single Query) - top 6:
1. Fy2024 Company Sales Strategy | teams | 2023-04-15
2. Fy2024 Company Sales Strategy | teams | 2023-04-15
3. Wfh Policy Update May 2023 | teams | 2023-05-01
4. Wfh Policy Update May 2023 | teams | 2023-05-01
5. April Work From Home Update | teams | 2022-04-29
6. April Work From Home Update | teams | 2022-04-29

--- Multi Query ---
Retrieved docs: 4

Sources (Multi Query) - top 4:
1. Fy2024 Company Sales Strategy | teams | 2023-04-15
2. Wfh Policy Update May 2023 | teams | 2023-05-01
3. New Employee Onboarding Guide | github | 2026-02-27T17:06:53.229535
4. April Work From Home Update | teams | 2022-04-29


--- Single Query Answer ---
Summary: The sales strategy for fiscal year 2024 aims to increase revenue by 20%, expand market share by 15%, retain 95% of existing customers, and launch two new products/services. The focus areas include targeting high-growth industries, strengthening c

## Manual multi-query generator + dedupe + retrieval budget

In [21]:
# =========================
# Iteration 2: Manual multi-query expansion + dedupe + retrieval budget
# =========================

from langchain_core.output_parsers import StrOutputParser

MANUAL_MULTI_QUERY_PROMPT = ChatPromptTemplate.from_template("""
You are generating search queries for a retriever.
Create 5 diverse search queries that could retrieve relevant passages.
- Use different phrasing and synonyms.
- Include 1 query that is very short (2-5 words).
- Include 1 query that is very specific (with constraints).
Return ONE query per line.

User question:
{question}

Queries:
""")

def dedupe_docs(docs):
    """Dedupe by (name + first 200 chars) to reduce repeated chunks."""
    seen = set()
    unique = []
    for d in docs:
        name = d.metadata.get("name") or d.metadata.get("source") or "unknown"
        key = (name, d.page_content[:200])
        if key not in seen:
            seen.add(key)
            unique.append(d)
    return unique

def manual_multiquery_rag(question: str, per_query_k: int = 3, max_docs: int = 10) -> str:
    query_chain = MANUAL_MULTI_QUERY_PROMPT | llm | StrOutputParser()
    raw = query_chain.invoke({"question": question})

    queries = [q.strip("-• \t") for q in raw.split("\n") if q.strip()]
    print("\nGenerated Queries (manual):")
    print("--------------------------")
    for q in queries:
        print("-", q)
    print("--------------------------")

    # Retrieve for each generated query using base retriever (NOT mq_retriever)
    gathered = []
    for q in queries:
        docs = base_retriever.invoke(q)
        gathered.extend(docs[:per_query_k])

    # Dedupe and cap (budget control)
    unique_docs = dedupe_docs(gathered)[:max_docs]

    print(f"\nDocs gathered: {len(gathered)} | After dedupe+cap: {len(unique_docs)}")
    print_sources(unique_docs, label="Manual MultiQuery (deduped)", max_items=10)

    # If retrieval is empty, return an honest response (helps prevent hallucination)
    if not unique_docs:
        return "I could not find relevant context in the indexed documents to answer this question."

    context = safe_combine_docs(unique_docs)
    return rag_chain.invoke({"question": question, "context": context})

question = "Explain <PUT A REAL TOPIC FROM YOUR DATA HERE> and list 3 key points."

print("\n==============================")
print("Iteration 2: Manual multi-query run")
print("==============================")
print("\n---- Answer ----")
print(manual_multiquery_rag(question))


Iteration 2: Manual multi-query run

---- Answer ----

Generated Queries (manual):
--------------------------
- 1. "Overview of <PUT A REAL TOPIC FROM YOUR DATA HERE>"
- 2. "What are the main aspects of <PUT A REAL TOPIC FROM YOUR DATA HERE>?"
- 3. "Key components of <PUT A REAL TOPIC FROM YOUR DATA HERE>"
- 4. "Define <PUT A REAL TOPIC FROM YOUR DATA HERE> and its significance"
- 5. "Characteristics of <PUT A REAL TOPIC FROM YOUR DATA HERE> explained"
--------------------------

Docs gathered: 15 | After dedupe+cap: 3

Sources (Manual MultiQuery (deduped)) - top 3:
1. Wfh Policy Update May 2023 | teams | 2023-05-01
2. Fy2024 Company Sales Strategy | teams | 2023-04-15
3. New Employee Onboarding Guide | github | 2026-02-27T17:06:53.229535
Explain the "WFH Policy Update May 2023" and list 3 key points:

1. Starting May 1, 2023, employees will be required to work from the office three days a week, with two days designated for remote work.
2. Employees need to communicate with their supe

## Lab Conclusion **Chatbot** with Multi-Query Retriever (Key Takeaways)

### What this lab was about

This lab showed how to build a simple RAG-style chatbot where the model answers questions using retrieved document passages, rather than relying on its own memory. The key focus was using a **Multi-Query Retriever** to improve retrieval quality.

---

### Core idea: why Multi-Query Retriever exists

A single user question is often not the best search query. Users may phrase things vaguely, use uncommon wording, or omit keywords that exist in the documents.

**Multi-Query Retriever improves recall** by:

* taking the user question,
* generating multiple alternative search queries (paraphrases, synonyms, more specific variants),
* retrieving documents for each query,
* combining the results into a richer context set for answering.

This increases the chance of retrieving relevant passages, especially when the dataset is diverse or terminology differs across documents.

---

### The RAG pipeline we built (end-to-end)

1. **Load documents** (source content + metadata)
2. **Split into passages** (chunking improves retrieval specificity and fits context limits)
3. **Store embeddings in Elasticsearch** (vector search over chunks)
4. **Retrieve relevant chunks** using:

   * a single-query retriever (baseline), and/or
   * a multi-query retriever (improved recall)
5. **Combine retrieved chunks into context**
6. **Answer using an LLM** with the retrieved context (grounding)

Key point: the chatbot’s answer quality is directly tied to the quality of retrieved context.

---

### Metadata matters (and why we had to fix it)

To index cleanly and keep results interpretable, each chunk needed consistent metadata fields:

* `name`
* `summary`
* `url`
* `category`
* `updated_at`

We also had to enforce valid values (especially `updated_at`) because Elasticsearch may map it as a `date`. If any document had `updated_at = None`, indexing failed.

Key takeaway: **consistent metadata prevents indexing errors and supports better debugging and transparency**.

---

### What we learned from the “iterations” requirement

We created new iterations at the bottom of the notebook to demonstrate mastery:

1. **Single-query vs Multi-query comparison**

   * Shows the baseline retrieval vs improved retrieval
   * Makes it obvious when MultiQuery improves coverage

2. **Manual multi-query expansion (with dedupe + retrieval budget)**

   * Demonstrates what MultiQueryRetriever is doing internally
   * Adds practical controls:

     * deduplication (avoid repeated chunks)
     * retrieval budget (avoid flooding the context window)

Key takeaway: MultiQuery is powerful, but you still need guardrails for context size and repetition.

---

### Common failure mode (and what it teaches)

If the indexed documents do not contain the topic the user asks about, the retriever cannot supply relevant context. In that case, the best behavior is an honest response like:

* “The context does not contain information to answer this.”

Key takeaway: **RAG systems are only as good as their data coverage**. Retrieval won’t invent knowledge that isn’t indexed.

---

### Final takeaway

* **RAG = Retrieval + Generation**: the LLM answers based on retrieved passages.
* **Multi-Query Retriever improves recall** by generating multiple search queries.
* **Chunking + clean metadata** are required for reliable indexing and retrieval.
* **Dedupe + retrieval budget** are practical controls that improve real-world performance.
* The best chatbot behavior is grounded: answer only when context supports it, otherwise say so.
